# Paper Trading System - Core & Engine Architecture Guide

**Phase 1 Tutorial**: Complete guide to the event-driven trading infrastructure

**Author**: Development Team  
**Date**: October 24, 2025  
**Version**: 1.0  

---

## Table of Contents

1. [Introduction](#introduction)
2. [Architecture Overview](#architecture)
3. [Part 1: Core Components](#part1)
   - Event System
   - Order Model
   - Position Model
4. [Part 2: Engine Components](#part2)
   - Order Management System
   - Portfolio Manager
   - Risk Manager
5. [Part 3: Integration Examples](#part3)
   - Complete Trading System
   - Market Making Simulation
6. [Part 4: Advanced Topics](#part4)
   - Performance Metrics with PLUTUS
   - Error Handling
   - Best Practices

---

## 1. Introduction <a name="introduction"></a>

This notebook provides a comprehensive guide to the **Paper Trading System** infrastructure built in Phase 1. The system is designed for algorithmic trading with a focus on:

- **Event-Driven Architecture**: Decoupled components communicating through events
- **Real-Time Portfolio Management**: Track positions and PnL in real-time
- **Risk Management**: Pre-trade validation and portfolio monitoring
- **Performance Analytics**: Integration with PLUTUS for comprehensive metrics

### Key Benefits

✅ **Modularity**: Components can be developed and tested independently  
✅ **Scalability**: Easy to add new strategies or data sources  
✅ **Testability**: High test coverage (96%) ensures reliability  
✅ **Production-Ready**: Clean code with comprehensive documentation  

### Prerequisites

- Python 3.13+
- Understanding of basic trading concepts
- Familiarity with Python dataclasses and type hints

### Setup

First, let's set up our environment and import necessary modules.

In [ ]:
# Add project root to path
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import core modules
from decimal import Decimal
from datetime import datetime
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print("✅ Environment setup complete")
print(f"📁 Project root: {project_root}")

## 2. Architecture Overview <a name="architecture"></a>

### System Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Paper Trading System                      │
└─────────────────────────────────────────────────────────────┘
                              │
                              ▼
                    ┌─────────────────┐
                    │    EventBus     │ ◄─── Central event dispatcher
                    └─────────────────┘
                              │
          ┌───────────────────┼───────────────────┐
          ▼                   ▼                   ▼
  ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
  │     OMS      │   │  Portfolio   │   │     Risk     │
  │  (Orders)    │   │  (Positions) │   │ (Validation) │
  └──────────────┘   └──────────────┘   └──────────────┘
          │                   │                   │
          └───────────────────┴───────────────────┘
                              │
                              ▼
                    ┌─────────────────┐
                    │  PLUTUS Metrics │
                    └─────────────────┘
```

### Event Flow

```
Market Data → Signal Event → Order Creation → Fill Event → Position Update → PnL Calculation
```

### Core Principles

1. **Publish-Subscribe Pattern**: Components subscribe to events they care about
2. **Single Responsibility**: Each component has one clear purpose
3. **Type Safety**: Full type hints throughout the codebase
4. **Decimal Precision**: All monetary calculations use Decimal type

## 3. Part 1: Core Components <a name="part1"></a>

The `core` module provides fundamental building blocks:
- **Enumerations**: Type-safe constants
- **Events**: Data structures for system communication
- **Order**: Order lifecycle management
- **Position**: Position tracking and PnL calculation

### 3.1 Event System

The event system is the heart of our architecture. It enables **loose coupling** between components.

In [ ]:
from core.event import EventBus, MarketDataEvent, SignalEvent, OrderEvent, FillEvent, TimeEvent
from core.enums import EventType

# Create event bus
bus = EventBus()

print("✅ EventBus created")
print(f"Available event types: {[e.value for e in EventType]}")

#### Event Types

| Event Type | Purpose | Published By | Consumed By |
|-----------|---------|--------------|-------------|
| **MARKET_DATA** | Price updates | Data feed | Portfolio, Strategy |
| **SIGNAL** | Strategy signals | Strategy | OMS |
| **ORDER** | Order state changes | OMS | Logging, Analytics |
| **FILL** | Order executions | Execution | OMS, Portfolio |
| **TIME** | Scheduled events | Scheduler | Portfolio |
| **SYSTEM** | System events | Any | Any |

#### Example: Publishing and Subscribing to Events

In [ ]:
# Create a handler function
received_events = []

def market_data_handler(event):
    """Handle market data events"""
    received_events.append(event)
    print(f"📊 Market Data: {event.contract} @ {event.price}")

# Subscribe to market data events
bus.subscribe(EventType.MARKET_DATA, market_data_handler)

# Create and publish event
market_event = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1250.5"),
    bid=Decimal("1250.0"),
    ask=Decimal("1251.0"),
    spread=Decimal("1.0"),
    volume=1000
)

bus.publish(market_event)
bus.process_events()

print(f"\n✅ Processed {len(received_events)} event(s)")

#### Multiple Handlers

Multiple handlers can subscribe to the same event type:

In [ ]:
# Add another handler
def price_logger(event):
    """Log price information"""
    print(f"📝 Logged: {event.contract} Bid={event.bid} Ask={event.ask} Spread={event.spread}")

bus.subscribe(EventType.MARKET_DATA, price_logger)

# Publish another event
market_event2 = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F2M",
    price=Decimal("1260.0"),
    bid=Decimal("1259.5"),
    ask=Decimal("1260.5"),
    spread=Decimal("1.0")
)

bus.publish(market_event2)
bus.process_events()

print("\n✅ Both handlers executed!")

### 3.2 Order Model

The Order class represents a single trading order with complete lifecycle tracking.

In [ ]:
from core.order import Order
from core.enums import OrderSide, OrderStatus

# Create an order
order = Order(
    contract="VN30F1M",
    side=OrderSide.BID,
    price=Decimal("1250.0"),
    quantity=1
)

print("📋 Order Created:")
print(f"   Order ID: {order.order_id[:8]}...")
print(f"   Contract: {order.contract}")
print(f"   Side: {order.side.value}")
print(f"   Price: {order.price}")
print(f"   Quantity: {order.quantity}")
print(f"   Status: {order.status.value}")
print(f"   Created: {order.created_at.strftime('%Y-%m-%d %H:%M:%S')}")

#### Order Lifecycle

```
CREATED → PENDING_SUBMIT → SUBMITTED → PARTIALLY_FILLED → FILLED
                                     ↓
                                  CANCELLED
                                     ↓
                                  REJECTED
```

In [ ]:
# Simulate order lifecycle
print("\n📈 Order Lifecycle Demo:")
print(f"Initial status: {order.status.value}")
print(f"Is active? {order.is_active()}")
print(f"Can cancel? {order.can_cancel()}")

# Submit order
order.status = OrderStatus.SUBMITTED
order.submitted_at = datetime.now()
print(f"\nAfter submission: {order.status.value}")
print(f"Is active? {order.is_active()}")

# Partial fill
order.status = OrderStatus.PARTIALLY_FILLED
order.filled_quantity = 0
print(f"\nPartially filled: {order.status.value}")
print(f"Unfilled quantity: {order.get_unfilled_quantity()}")

# Complete fill
order.status = OrderStatus.FILLED
order.filled_quantity = 1
order.filled_price = Decimal("1250.5")
order.filled_at = datetime.now()
print(f"\nFilled: {order.status.value}")
print(f"Fill price: {order.filled_price}")
print(f"Is terminal? {order.is_terminal()}")
print(f"Can cancel? {order.can_cancel()}")

### 3.3 Position Model

The Position class tracks holdings and calculates PnL for a single contract.

In [ ]:
from core.position import Position

# Create a position (initially flat)
position = Position(
    contract="VN30F1M",
    quantity=0,
    average_price=Decimal("0")
)

print("📊 Position Created:")
print(f"   Contract: {position.contract}")
print(f"   Quantity: {position.quantity}")
print(f"   Is flat? {position.is_flat()}")

#### Simulating Trades

Let's simulate buying and selling to see PnL calculation:

In [ ]:
# Simulate buy at 1250
position.quantity = 1
position.average_price = Decimal("1250")

print("\n🟢 After buying 1 contract @ 1250:")
print(f"   Quantity: {position.quantity}")
print(f"   Average price: {position.average_price}")
print(f"   Is long? {position.is_long()}")

# Update market price to 1260
position.update_unrealized_pnl(Decimal("1260"))

print(f"\n📈 Market price moves to 1260:")
print(f"   Unrealized PnL: {position.unrealized_pnl:,.2f} VND")
print(f"   Calculation: (1260 - 1250) × 1 × 100 = {position.unrealized_pnl}")

# Add fees
position.total_fees = Decimal("20")

print(f"\n💰 After fees:")
print(f"   Realized PnL: {position.realized_pnl}")
print(f"   Unrealized PnL: {position.unrealized_pnl}")
print(f"   Fees: {position.total_fees}")
print(f"   Total PnL: {position.total_pnl():,.2f} VND")

#### Long vs Short Positions

In [ ]:
# Create short position
short_position = Position(
    contract="VN30F2M",
    quantity=-2,  # Negative = short
    average_price=Decimal("1260")
)

print("🔴 Short Position (sold at 1260):")
print(f"   Quantity: {short_position.quantity}")
print(f"   Is short? {short_position.is_short()}")

# Price drops to 1240 (profit for short)
short_position.update_unrealized_pnl(Decimal("1240"))

print(f"\n📉 Market price drops to 1240 (profitable for short):")
print(f"   Unrealized PnL: {short_position.unrealized_pnl:,.2f} VND")
print(f"   Calculation: (1260 - 1240) × 2 × 100 = {short_position.unrealized_pnl}")

# Price rises to 1270 (loss for short)
short_position.update_unrealized_pnl(Decimal("1270"))

print(f"\n📈 Market price rises to 1270 (loss for short):")
print(f"   Unrealized PnL: {short_position.unrealized_pnl:,.2f} VND")
print(f"   Calculation: (1260 - 1270) × 2 × 100 = {short_position.unrealized_pnl}")

## 4. Part 2: Engine Components <a name="part2"></a>

The `engine` module provides the trading system components:
- **OrderManager (OMS)**: Order lifecycle management
- **PortfolioManager**: Position tracking and PnL
- **RiskManager**: Pre-trade validation

### 4.1 Order Management System (OMS)

The OMS manages the complete order lifecycle.

In [ ]:
from engine.oms import OrderManager

# Create OMS
oms = OrderManager(bus, risk_manager=None)

print("✅ OMS Created")
print(f"Total orders: {len(oms.orders)}")
print(f"Active orders: {len(oms.active_orders)}")

#### Creating and Submitting Orders

In [ ]:
# Create orders
bid_order = oms.create_order(
    contract="VN30F1M",
    side=OrderSide.BID,
    price=Decimal("1248"),
    quantity=1
)

ask_order = oms.create_order(
    contract="VN30F1M",
    side=OrderSide.ASK,
    price=Decimal("1253"),
    quantity=1
)

print("📋 Orders Created:")
print(f"   Bid: {bid_order.side.value} @ {bid_order.price}")
print(f"   Ask: {ask_order.side.value} @ {ask_order.price}")

# Submit orders
oms.submit_order(bid_order)
oms.submit_order(ask_order)

print(f"\n✅ Orders submitted")
print(f"Active orders: {len(oms.get_active_orders())}")

#### OMS Statistics

In [ ]:
stats = oms.get_statistics()

print("📊 OMS Statistics:")
print(f"   Total orders: {stats['total_orders']}")
print(f"   Active: {stats['active_orders']}")
print(f"   Filled: {stats['filled_orders']}")
print(f"   Cancelled: {stats['cancelled_orders']}")
print(f"   Rejected: {stats['rejected_orders']}")

#### Event-Driven Order Updates

OMS automatically updates orders when receiving signal events:

In [ ]:
# Subscribe OMS to signals
bus.subscribe(EventType.SIGNAL, oms.on_signal_event)

# Create signal event (from strategy)
signal = SignalEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    signal_type="UPDATE_BID_ASK",
    bid_price=Decimal("1249"),
    ask_price=Decimal("1254"),
    reason="TIME_ELAPSED"
)

print("📡 Publishing signal event...")
bus.publish(signal)
bus.process_events()

# Check updated orders
active = oms.get_active_orders_by_contract("VN30F1M")
print(f"\n✅ Orders updated automatically")
print(f"Active orders: {len(active)}")
for order in active:
    print(f"   {order.side.value} @ {order.price}")

### 4.2 Portfolio Manager

The Portfolio Manager tracks all positions and calculates real-time PnL.

In [ ]:
from engine.portfolio import PortfolioManager

# Create portfolio with initial capital
portfolio = PortfolioManager(bus, Decimal("500000"))

print("💼 Portfolio Created:")
print(f"   Initial capital: {portfolio.initial_capital:,} VND")
print(f"   Cash: {portfolio.cash:,} VND")
print(f"   NAV: {portfolio.calculate_nav():,} VND")

#### Simulating Fills

Let's simulate order fills and see portfolio updates:

In [ ]:
# Subscribe portfolio to fills
bus.subscribe(EventType.FILL, portfolio.on_fill_event)
bus.subscribe(EventType.MARKET_DATA, portfolio.on_market_data)

# Simulate buy fill
buy_fill = FillEvent(
    timestamp=datetime.now(),
    order_id=bid_order.order_id,
    contract="VN30F1M",
    side="BID",
    fill_price=Decimal("1249"),
    fill_quantity=1,
    fee=Decimal("20")
)

print("🟢 Processing buy fill...")
bus.publish(buy_fill)
bus.process_events()

# Check position
pos = portfolio.get_position("VN30F1M")
print(f"\n📊 Position after buy:")
print(f"   Quantity: {pos.quantity}")
print(f"   Average price: {pos.average_price}")
print(f"   Cash: {portfolio.cash:,} VND")
print(f"   NAV: {portfolio.calculate_nav():,} VND")

#### Market Price Updates

In [ ]:
# Market price moves up
market_update = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1260"),
    bid=Decimal("1259"),
    ask=Decimal("1261"),
    spread=Decimal("2")
)

print("📈 Market price moves to 1260...")
bus.publish(market_update)
bus.process_events()

print(f"\n💰 PnL Update:")
print(f"   Unrealized PnL: {pos.unrealized_pnl:,} VND")
print(f"   Total PnL: {pos.total_pnl():,} VND")
print(f"   NAV: {portfolio.calculate_nav():,} VND")

#### Portfolio Summary

In [ ]:
summary = portfolio.get_summary()

print("💼 Portfolio Summary:")
print(f"   Cash: {summary['cash']:,.2f} VND")
print(f"   NAV: {summary['nav']:,.2f} VND")
print(f"   Total Return: {summary['total_return']:.2f}%")

print(f"\n📊 Positions:")
for contract, pos_data in summary['positions'].items():
    print(f"   {contract}:")
    print(f"      Quantity: {pos_data['quantity']}")
    print(f"      Avg Price: {pos_data['average_price']:.2f}")
    print(f"      Unrealized PnL: {pos_data['unrealized_pnl']:,.2f}")
    print(f"      Total PnL: {pos_data['total_pnl']:,.2f}")

#### Margin Management

In [ ]:
# Check available margin
available = portfolio.get_available_margin("VN30F1M", Decimal("1250"))

print("💵 Margin Analysis:")
print(f"   Current NAV: {portfolio.calculate_nav():,} VND")
print(f"   Price: 1,250")
print(f"   Margin per contract: 1,250 × 100 × 0.17 = 21,250 VND")
print(f"   Available margin: {available} contracts")
print(f"\n   Can place {available} more contracts at current price")

### 4.3 Risk Manager

The Risk Manager validates orders before submission.

In [ ]:
from engine.risk import RiskManager

# Create risk manager
risk = RiskManager(portfolio)

print("🛡️ Risk Manager Created")

#### Order Validation

In [ ]:
# Valid order
valid_order = Order(
    contract="VN30F1M",
    side=OrderSide.BID,
    price=Decimal("1250"),
    quantity=1
)

if risk.validate_order(valid_order):
    print("✅ Order passed validation")
else:
    print("❌ Order rejected")

# Invalid order (too many contracts)
invalid_order = Order(
    contract="VN30F1M",
    side=OrderSide.BID,
    price=Decimal("1250"),
    quantity=100  # Too many!
)

print(f"\nValidating large order ({invalid_order.quantity} contracts)...")
if risk.validate_order(invalid_order):
    print("✅ Order passed validation")
else:
    print("❌ Order rejected - insufficient margin")

# Invalid price
invalid_price_order = Order(
    contract="VN30F1M",
    side=OrderSide.BID,
    price=Decimal("0"),  # Invalid!
    quantity=1
)

print(f"\nValidating order with price={invalid_price_order.price}...")
if risk.validate_order(invalid_price_order):
    print("✅ Order passed validation")
else:
    print("❌ Order rejected - invalid price")

#### Portfolio Health Check

In [ ]:
# Check margin requirement
is_healthy = risk.check_margin_requirement()

print("🏥 Portfolio Health Check:")
if is_healthy:
    print("   ✅ Portfolio meets margin requirements")
else:
    print("   ⚠️  Margin call - liquidation needed")

# Calculate requirements
nav = portfolio.calculate_nav()
required_margin = Decimal("0")
for position in portfolio.positions.values():
    if position.quantity != 0:
        price = portfolio.current_prices.get(position.contract, position.average_price)
        required_margin += abs(position.quantity) * price * Decimal('100') * Decimal('0.17')

print(f"\n   NAV: {nav:,.2f} VND")
print(f"   Required margin: {required_margin:,.2f} VND")
print(f"   Safety buffer: {(nav - required_margin):,.2f} VND")

## 5. Part 3: Integration Examples <a name="part3"></a>

Let's build a complete trading system integrating all components.

### 5.1 Complete Trading System Setup

In [ ]:
# Create fresh instances
bus = EventBus()
portfolio = PortfolioManager(bus, Decimal("1000000"))  # 1M VND capital
risk = RiskManager(portfolio)
oms = OrderManager(bus, risk_manager=risk)

# Subscribe to events
bus.subscribe(EventType.SIGNAL, oms.on_signal_event)
bus.subscribe(EventType.FILL, oms.on_fill_event)
bus.subscribe(EventType.FILL, portfolio.on_fill_event)
bus.subscribe(EventType.MARKET_DATA, portfolio.on_market_data)

print("🚀 Complete Trading System Initialized")
print(f"   Capital: {portfolio.initial_capital:,} VND")
print(f"   Risk management: Enabled")
print(f"   Event subscriptions: Configured")

### 5.2 Market Making Simulation

Simulate a complete market making cycle:

In [ ]:
print("\n" + "="*60)
print("  MARKET MAKING SIMULATION")
print("="*60)

# Step 1: Market data arrives
print("\n📊 Step 1: Market data arrives")
market_data = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1250"),
    bid=Decimal("1249.5"),
    ask=Decimal("1250.5"),
    spread=Decimal("1.0")
)
bus.publish(market_data)
bus.process_events()
print(f"   Market: {market_data.contract} @ {market_data.price}")

# Step 2: Strategy generates signal
print("\n🎯 Step 2: Strategy generates signal")
signal = SignalEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    signal_type="UPDATE_BID_ASK",
    bid_price=Decimal("1248"),  # Below market
    ask_price=Decimal("1253"),  # Above market
    reason="INITIAL"
)
bus.publish(signal)
bus.process_events()
print(f"   Signal: BID @ {signal.bid_price}, ASK @ {signal.ask_price}")

# Step 3: Check orders
print("\n📋 Step 3: Orders created and submitted")
active = oms.get_active_orders_by_contract("VN30F1M")
for order in active:
    print(f"   {order.side.value} @ {order.price} [{order.status.value}]")

# Step 4: Bid order gets filled
print("\n💰 Step 4: Bid order filled (bought at 1248)")
bid_order = [o for o in active if o.side == OrderSide.BID][0]
fill = FillEvent(
    timestamp=datetime.now(),
    order_id=bid_order.order_id,
    contract="VN30F1M",
    side="BID",
    fill_price=Decimal("1248"),
    fill_quantity=1,
    fee=Decimal("20")
)
bus.publish(fill)
bus.process_events()

pos = portfolio.get_position("VN30F1M")
print(f"   Position: {pos.quantity} contract @ {pos.average_price}")
print(f"   Cash: {portfolio.cash:,} VND")

# Step 5: Market moves in our favor
print("\n📈 Step 5: Market moves up to 1255")
market_data2 = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1255")
)
bus.publish(market_data2)
bus.process_events()

print(f"   Unrealized PnL: {pos.unrealized_pnl:,} VND")
print(f"   NAV: {portfolio.calculate_nav():,} VND")

# Step 6: Ask order gets filled
print("\n💵 Step 6: Ask order filled (sold at 1253)")
ask_order = [o for o in oms.get_active_orders() if o.side == OrderSide.ASK][0]
sell_fill = FillEvent(
    timestamp=datetime.now(),
    order_id=ask_order.order_id,
    contract="VN30F1M",
    side="ASK",
    fill_price=Decimal("1253"),
    fill_quantity=1,
    fee=Decimal("20")
)
bus.publish(sell_fill)
bus.process_events()

print(f"   Position: {pos.quantity} contracts (flat)")
print(f"   Realized PnL: {pos.realized_pnl:,} VND")
print(f"   Cash: {portfolio.cash:,} VND")

# Final summary
print("\n" + "="*60)
print("  FINAL SUMMARY")
print("="*60)
summary = portfolio.get_summary()
print(f"   Total Return: {summary['total_return']:.4f}%")
print(f"   Final NAV: {summary['nav']:,.2f} VND")
print(f"   Profit/Loss: {summary['nav'] - 1000000:,.2f} VND")
print(f"\n   Trade: Buy @ 1248, Sell @ 1253")
print(f"   Gross profit: (1253 - 1248) × 100 = 500 VND")
print(f"   Fees: 20 + 20 = 40 VND")
print(f"   Net profit: 500 - 40 = 460 VND ✅")

### 5.3 Multiple Trades Simulation

In [ ]:
# Reset system
bus = EventBus()
portfolio = PortfolioManager(bus, Decimal("1000000"))
risk = RiskManager(portfolio)
oms = OrderManager(bus, risk_manager=risk)

bus.subscribe(EventType.SIGNAL, oms.on_signal_event)
bus.subscribe(EventType.FILL, oms.on_fill_event)
bus.subscribe(EventType.FILL, portfolio.on_fill_event)
bus.subscribe(EventType.MARKET_DATA, portfolio.on_market_data)

print("🔄 Running multiple trade cycles...\n")

trades = [
    {"buy": Decimal("1248"), "sell": Decimal("1253"), "market": Decimal("1250")},
    {"buy": Decimal("1252"), "sell": Decimal("1257"), "market": Decimal("1254")},
    {"buy": Decimal("1255"), "sell": Decimal("1260"), "market": Decimal("1257")},
]

for i, trade in enumerate(trades, 1):
    print(f"Trade {i}:")
    
    # Market data
    market_data = MarketDataEvent(
        timestamp=datetime.now(),
        contract="VN30F1M",
        price=trade["market"]
    )
    bus.publish(market_data)
    
    # Signal
    signal = SignalEvent(
        timestamp=datetime.now(),
        contract="VN30F1M",
        signal_type="UPDATE_BID_ASK",
        bid_price=trade["buy"],
        ask_price=trade["sell"],
        reason="TIME_ELAPSED"
    )
    bus.publish(signal)
    bus.process_events()
    
    # Get orders
    active = oms.get_active_orders_by_contract("VN30F1M")
    bid_order = [o for o in active if o.side == OrderSide.BID][0]
    ask_order = [o for o in active if o.side == OrderSide.ASK][0]
    
    # Fill buy
    buy_fill = FillEvent(
        order_id=bid_order.order_id,
        contract="VN30F1M",
        side="BID",
        fill_price=trade["buy"],
        fill_quantity=1,
        fee=Decimal("20")
    )
    bus.publish(buy_fill)
    
    # Fill sell
    sell_fill = FillEvent(
        order_id=ask_order.order_id,
        contract="VN30F1M",
        side="ASK",
        fill_price=trade["sell"],
        fill_quantity=1,
        fee=Decimal("20")
    )
    bus.publish(sell_fill)
    bus.process_events()
    
    pos = portfolio.get_position("VN30F1M")
    gross = (trade["sell"] - trade["buy"]) * 100
    net = gross - 40
    print(f"   Buy @ {trade['buy']}, Sell @ {trade['sell']}")
    print(f"   Gross: {gross} VND, Net: {net} VND")
    print(f"   Cumulative PnL: {pos.realized_pnl} VND\n")

# Final summary
summary = portfolio.get_summary()
print("="*50)
print(f"Final NAV: {summary['nav']:,.2f} VND")
print(f"Total Return: {summary['total_return']:.4f}%")
print(f"Total Profit: {summary['nav'] - 1000000:,.2f} VND")
print(f"Trades executed: {len(trades)}")
print(f"Average profit per trade: {(summary['nav'] - 1000000) / len(trades):,.2f} VND")

## 6. Part 4: Advanced Topics <a name="part4"></a>

### 6.1 Daily Settlement and Performance Metrics

Simulate daily settlement to track performance over time:

In [ ]:
# Subscribe to time events
bus.subscribe(EventType.TIME, portfolio.on_time_event)

# Trigger daily settlement
time_event = TimeEvent(
    timestamp=datetime.now(),
    event_name="DAILY_SETTLEMENT",
    date=datetime.now()
)

print("📅 Processing daily settlement...")
bus.publish(time_event)
bus.process_events()

print(f"\nDaily returns recorded: {len(portfolio.daily_returns)}")
print(f"Daily NAV history: {len(portfolio.daily_nav)}")

if len(portfolio.daily_returns) > 0:
    print(f"\nLast daily return: {portfolio.daily_returns[-1] * 100:.4f}%")
    print(f"Current NAV: {portfolio.daily_nav[-1]:,.2f} VND")

### 6.2 Error Handling

The system gracefully handles errors:

In [ ]:
print("🛡️ Testing error handling...\n")

# Test 1: Unknown order fill
print("Test 1: Fill for unknown order")
unknown_fill = FillEvent(
    order_id="unknown-order-id",
    contract="VN30F1M",
    side="BID",
    fill_price=Decimal("1250"),
    fill_quantity=1,
    fee=Decimal("20")
)
bus.publish(unknown_fill)
bus.process_events()
print("   ✅ System continued (error logged)\n")

# Test 2: Cancel non-existent order
print("Test 2: Cancel non-existent order")
result = oms.cancel_order("non-existent-id")
print(f"   Result: {result}")
print("   ✅ System handled gracefully\n")

# Test 3: Handler exception
print("Test 3: Handler with exception")
def bad_handler(event):
    raise ValueError("Intentional error")

def good_handler(event):
    print("   Good handler still executed")

bus.subscribe(EventType.MARKET_DATA, bad_handler)
bus.subscribe(EventType.MARKET_DATA, good_handler)

market_event = MarketDataEvent(
    timestamp=datetime.now(),
    contract="VN30F1M",
    price=Decimal("1250")
)
bus.publish(market_event)
bus.process_events()
print("   ✅ Exception caught, other handlers continued\n")

### 6.3 Best Practices

#### 1. Always use Decimal for monetary values

In [ ]:
# ✅ Good
price = Decimal("1250.50")
print(f"Good: {price} (type: {type(price).__name__})")

# ❌ Bad (floating point errors)
price_float = 1250.50
print(f"Bad: {price_float} (type: {type(price_float).__name__})")

# Demonstration of float precision issues
print(f"\nFloat issue: 0.1 + 0.2 = {0.1 + 0.2}")
print(f"Decimal: Decimal('0.1') + Decimal('0.2') = {Decimal('0.1') + Decimal('0.2')}")

#### 2. Always process events after publishing

In [ ]:
# ✅ Good
bus.publish(event)
bus.process_events()  # Events are dispatched

# ❌ Bad
# bus.publish(event)
# # Forgot to call process_events() - handlers won't be called!

print("✅ Always call process_events() after publishing")

#### 3. Use event-driven communication

In [ ]:
# ✅ Good (event-driven)
signal = SignalEvent(
    contract="VN30F1M",
    bid_price=Decimal("1248"),
    ask_price=Decimal("1253")
)
bus.publish(signal)
bus.process_events()

# ❌ Bad (direct coupling)
# oms.on_signal_event(signal)  # Don't call directly!

print("✅ Use event bus for component communication")

#### 4. Check order status before operations

In [ ]:
order = Order(
    contract="VN30F1M",
    side=OrderSide.BID,
    price=Decimal("1250"),
    quantity=1
)

# ✅ Good
if order.can_cancel():
    oms.cancel_order(order.order_id)
    print("✅ Order cancelled")
else:
    print("⚠️ Cannot cancel order in current state")

# Check status
if order.is_active():
    print("Order is active")
elif order.is_terminal():
    print("Order is in terminal state")

#### 5. Validate with risk manager

In [ ]:
order = Order(
    contract="VN30F1M",
    side=OrderSide.BID,
    price=Decimal("1250"),
    quantity=1
)

# ✅ Good
if risk.validate_order(order):
    oms.submit_order(order)
    print("✅ Order submitted after validation")
else:
    print("❌ Order rejected by risk manager")

# Even better: Use OMS with risk manager
oms_with_risk = OrderManager(bus, risk_manager=risk)
# Now validation happens automatically!
print("✅ OMS with automatic risk validation")

## Summary

### Key Takeaways

1. **Event-Driven Architecture**
   - Loose coupling between components
   - Easy to extend and maintain
   - Clear separation of concerns

2. **Order Management**
   - Complete lifecycle tracking
   - Automatic updates from signals
   - Event-driven communication

3. **Portfolio Management**
   - Real-time position tracking
   - Accurate PnL calculation
   - Margin management

4. **Risk Management**
   - Pre-trade validation
   - Portfolio health monitoring
   - Automatic rejection of invalid orders

### System Statistics

- **Test Coverage**: 96%
- **Production Code**: 1,100 lines
- **Test Code**: 1,538 lines
- **Modules**: 7 (4 core + 3 engine)
- **Test Execution**: 0.12 seconds

### Next Steps

1. **Phase 2**: Implement market making strategy
2. **Phase 2**: Add execution simulator
3. **Phase 2**: Historical data playback
4. **Phase 2**: End-to-end testing

### Resources

- [Core Module Guide](../core/README.md)
- [Engine Module Guide](../engine/README.md)
- [Week 1 Report](../markdown-docs/week-1-completion-report.md)
- [Week 2 Report](../markdown-docs/week-2-completion-report.md)
- [Full Specification](../internal-docs/paper-trading-spec.md)

---

## Exercise: Build Your Own Strategy

Try modifying the market making simulation to:

1. Adjust bid/ask spread based on inventory
2. Implement position limits
3. Add time-based order updates (every 15 seconds)
4. Calculate daily returns and Sharpe ratio

**Happy Trading! 🚀**